# 03 · End-to-end (E2E) training

The portfolio problem is a layer of the network; the exact layer and its gradient are in `mlm/e2e.py` and
`mlm/meansd.py` and are checked in `checks/test_e2e_layer.py`, `checks/test_meansd.py` and
`checks/test_e2e_runs.py`. Every E2E model starts from its trained two-stage version (Appendix A.2.2).

| Section | Paper | Runs |
|---|---|---|
| A | Table 2 / A.3 | E2E for Linear and NN1-NN5, long-only top decile; smoothing ζ chosen on validation |
| B | Table 3 | NN3 trained with costs of 10-40 bps (eq. 20) |
| C | our extensions | E2E from ridge, PCA20-NN3 and GMM experts; decision-aware PCA (learnable projection, penalty μ) |
| D | PAPER_ISSUES D1 | NN3 trained from scratch, to check the warm start is not doing the work |
| E | Table 4 / 5 | mean-SD E2E and PPP across risk aversion η, with and without 20 bps costs |

**Run each section only after its time estimate** (measured on one window first).

In [ ]:
from pathlib import Path

from mlm.data import rolling_windows
from mlm.e2e import E2EConfig, run_e2e
from mlm.models import MODELS

PANEL_DIR = Path("D:/ML_Markowitz_data/panel_cn")
PRED_DIR = Path("D:/ML_Markowitz_data/preds/cn")   # two-stage models (warm starts) and E2E outputs
windows = rolling_windows("2014-01-01", "2023-12-31")
ZETA_GRID = (0.05, 0.2, 1.0)   # smoothing, chosen per model and window on validation (PAPER_ISSUES A2)

## A. E2E for the paper's models (Tables 2 and A.3)

In [ ]:
run_e2e(MODELS, PANEL_DIR, windows, PRED_DIR, PRED_DIR, E2EConfig(zeta_rel=ZETA_GRID), tag='E2E')

## B. Transaction costs (Table 3): NN3 trained with each cost rate

In [ ]:
for bps in (10, 20, 30, 40):
    run_e2e(["NN3"], PANEL_DIR, windows, PRED_DIR, PRED_DIR, E2EConfig(gamma=bps / 1e4), tag=f"E2E-tc{bps}")

## C. E2E for our models, and decision-aware PCA

In [ ]:
run_e2e(["Ridge", "PCA20-NN3", "GMM-Experts"], PANEL_DIR, windows, PRED_DIR, PRED_DIR, E2EConfig(zeta_rel=ZETA_GRID), tag="E2E")
for mu in (0.1, 1.0, 10.0):   # strength of the pull toward the PCA loadings
    run_e2e(["PCA20-Linear", "PCA20-NN3"], PANEL_DIR, windows, PRED_DIR, PRED_DIR, E2EConfig(da_mu=mu), tag=f"DA{mu:g}")

## D. Warm start check: NN3 from scratch (first 8 windows)

In [ ]:
run_e2e(['NN3'], PANEL_DIR, windows[:8], PRED_DIR, PRED_DIR, E2EConfig(warm_start=False), tag='E2E-scratch')

## E. Risk aversion (Tables 4 and 5): mean-SD E2E and PPP

In [ ]:
ETAS = (0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 7.5, 10.0)
for eta in ETAS:
    lr = 1e-4 if eta < 1 else 1e-3   # as in Appendix A.2.2
    for gamma, g in ((0.0, "gross"), (0.002, "net")):
        run_e2e(["NN3"], PANEL_DIR, windows, PRED_DIR, PRED_DIR, E2EConfig(objective="meansd", eta=eta, gamma=gamma, lr=lr),
                tag=f"MSD-{g}-eta{eta:g}")
        run_e2e(["NN3"], PANEL_DIR, windows, PRED_DIR, PRED_DIR, E2EConfig(objective="ppp", eta=eta, gamma=gamma, lr=lr),
                tag=f"PPP-{g}-eta{eta:g}")